In [343]:
%load_ext autoreload
%autoreload 2

# Cabling Tools

## Imports

In [345]:
import math
from dataclasses import dataclass
from typing import Optional, List

import sys
from pathlib import Path

# Path to the directory holding the notebook
current_dir = Path.cwd()

# Point to parent (one level up) or root
parent_dir = current_dir.parent.parent

if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

# Now import directly from the parent folder/package

from ictdesigntools.cabling.optical import Wavelength

In [346]:
Wavelength

ictdesigntools.cabling.optical.Wavelength

## Optical Power Budget

The optical power budget calcalates the amount of available optical power across an optical cable channel within an optical fiber cable. The optical power budget calculation is the relative difference (dB) between a link's transmitter (TX) launch power (dBm) and receiver (RX) sensitivity level (dBm):

$$\text{Optical Power Budget (dB)} = \text{Optical TX Launch Power (dBm)} - \text{Optical RX Sensitivity Level (dBm)}$$

For this budget, all data is derived from the pair of optical transceivers used on the link. In most cases the transceivers will have the same symmetrical specifications and originate from the same manufacturer, but this isn't always true. Transceivers with symmetrical requirements may come from two different manufacturers, with slightly different performance specifications. In special applications like Gigabit Passive Optical Networks (GPON), the upstream transceiver in the optical line terminal (OLT) has an entirely different configuration than the downstream transceiver in the optical network terminal (ONT) and needs to be calculated as such.

Therefore, it's important to define the parameters that will be used on both sides. Each transceiver on the link can have a name to identify it by, such as "Transceiver A" and "Transceiver B". It requires a nominal TX power level (and optionally values for the maximum TX power and minimum TX power to more effectively account for the launch power range), as well as the RX sensitivity and overload levels to account for weak signal and saturation conditions that may degrade or destroy the link.



According to the OTT, typical TX launch power levels are as follows:
* LED launch into 8/125 fiber: -25dBm
* Last mile telecom links: -5dBm to -8dBm
* Long haul telecom links: 0 dBm to 5dBm
* Amplified telecom links: 5dBm to 27dBm

For RX sensitivity levels:
* Lower rate telecom links: -35dB
* Higher rate telecom links: -25dBm

Whenever possible, transceiver manufacturer specifications for power levels and sensitivity levels should be used.

The result of the calculation is the power budget, a value in (dB) representing the relative amount of signal strength between the transmitters and receivers of an optical link. For an optical link to stay connected, this value must not be exhuasted by loss across the link. In fact, it's important to define a safety margin for the power budget to factor in unallocated power loss from transmitter laser diode degradation over time, thermal drift, patch cable wear, or future emergency repair splices within the link. For this, a safety margin of 2dB or more should be designed into the power budget. It's also important to define a 

### Implementation Data Classes

In [339]:
wl_1490 = Wavelength("1490", nominal_nm=1490, min_nm=1480, max_nm=1500)
wl_1310 = Wavelength("1310", nominal_nm=1310, min_nm=1290, max_nm=1330)
print(wl_1490)
print(wl_1310)

Wavelength(name='1490', nominal_nm=1490, min_nm=1480, max_nm=1500)
Wavelength(name='1310', nominal_nm=1310, min_nm=1290, max_nm=1330)


In [335]:
# We'll assume the exact same transceivers on both sides of the link.

transceiver_a = OpticalTransceiver('A', 1310, -4.6, -4.5, -1, -2)
transceiver_b = OpticalTransceiver('B', 1550, -4.6, -4.5, -1, -2)

print(transceiver_a)
print(transceiver_b)

OpticalTransceiver(name='A', wavelenth=1310, tx_power_nominal_dbm=-4.6, tx_power_min_dbm=-4.5, tx_power_max_dbm=-1, rx_overload_dbm=-2, rx_sensitivity_dbm=-25)
OpticalTransceiver(name='B', wavelenth=1550, tx_power_nominal_dbm=-4.6, tx_power_min_dbm=-4.5, tx_power_max_dbm=-1, rx_overload_dbm=-2, rx_sensitivity_dbm=-25)


### Cable Channel and Optical Loss Budget Data Classes

## Optical Link Margin Calculator

The **optical link margin** calculation is the difference (in dB) between the optical power budget and the optical loss budget:

$$\text{Link Margin (dB)} = \text{Optical Power Budget (dB)} - \text{Optical Loss Budget (dB)}$$

Calculating the optical link margin is a multi-step process, typically beginning with calculating the optical power budget, then the optical loss budget, and finally, calculating the optical link margin.

For the **optical loss budget**, all data is derived from the elements of the cable channel(s) used on the link. Here the cable plant configuration determines the amount of loss within the channel(s). For that we will provide an `CableChannel` data class composed of cable spans, splices, and connectors to represent the cable channel.

The optical loss budget calculation is the sum of insertion loss (dB) of the cables, splices, and connectors within the channel:

$$\text{Optical Loss Budget (dB)} = \text{Total Cable Span Attenuation (dB)} + \text{Total Splice Attenuation (dB)} + \text{Total Connector Attenuation (dB)}$$

According to the OTT, typical insertion loss power levels are as follows:


In [313]:
span1 = CableSpan(1000)
print(span1)
print(span1.loss_db)
print(span1.loss_db_str)

CableSpan(cable_length_m=1000, cable_type='G.652', cable_loss_db_km=0.01)
0.01
0.01dB


In [311]:
span2 = CableSpan(10000)
print(span2)
print(span2.loss_db)
print(span2.loss_db_str)

CableSpan(cable_length_m=10000, cable_type='G.652', cable_loss_db_km=0.01)
0.1
0.1dB


In [326]:
span1

CableSpan(cable_length_m=1000, cable_type='G.652', cable_loss_db_km=0.01)

## Optical Power Budget Calculator [[Issue Link](https://github.com/eronlloyd/ictdesigntools/issues/2)]

In [218]:
def optical_power_budget(tx, rx):
    budget = tx.launch_power_dbm - rx.receive_sensitivity_dbm
    return PowerBudget(budget)

In [219]:
tx = Transmitter(launch_power_dbm=-5)
rx = Receiver(receive_sensitivity_dbm=-25)

power_budget = optical_power_budget(tx, rx)
print(power_budget)

PowerBudget(budget=20, safety_margin=2)


In [220]:
tx = Transmitter(launch_power_dbm=0)
rx = Receiver(receive_sensitivity_dbm=-25)

power_budget = optical_power_budget(tx, rx)
print(power_budget)

PowerBudget(budget=25, safety_margin=2)


### Optical Loss Budget Calculator

## Optical loss budget calculator; [issue link](https://github.com/eronlloyd/ictdesigntools/issues/1)

In [324]:
@dataclass(frozen=True)
class Cable:
    

@dataclass(frozen=True)
class Splice:
    splice_type: str='fusion'
    loss_db: float=0.1

    def __post_init__(self):
        if self.loss_db <= 0:
            raise ValueError("Splice loss must be greater than 0 dB.")

@dataclass(frozen=True)
class Connector:
    connector_type: str='LC/UPC'
    loss_db: float=0.5

    def __post_init__(self):
        if self.loss_db <= 0:
            raise ValueError("Connector loss must be greater than 0 dB.")

@dataclass(frozen=True)
class LossBudget:
    budget: int

IndentationError: expected an indented block after class definition on line 2 (3554753782.py, line 5)

In [325]:
cables = [Cable(cable_length_m=40, cable_loss_db_km=0.37)]
splices = [Splice() for splice in range(21)]
connectors = [Connector() for connector in range(2)]

loss_budget = optical_loss_budget(cables, splices, connectors)
loss_budget

TypeError: Splice.__init__() missing 1 required positional argument: 'name'

## Optical Link Margin Calculator

In [236]:
# TODO: Call it something more appropriate

def optical_link_margin(power_budget, loss_budget):
    print(f'Power budget is {round(power_budget.budget, 3)}dB, and loss budget is {round(loss_budget.budget, 3)}dB.')
    loss = power_budget - loss_budget
    margin = loss - power_budget.safety_margin
    print(f'The net budget is {round(loss, 3)}dB and the margin of the safety factor is {round(margin, 3)}dB')
    if loss > power_budget.safety_margin:
        print(f'Budget is {round(margin, 3)}dB over the required {power_budget.safety_margin}dB safety margin.')
        return loss
    else:
        print(f'Budget is {round(margin, 3)}dB under the required {power_budget.safety_margin}dB safety margin.')
        raise Exception

### General Link Margin Example

In [237]:
tx = Transmitter(launch_power_dbm=-4.9)
rx = Receiver(receive_sensitivity_dbm=-25)

power_budget = optical_power_budget(tx, rx)
print(power_budget)
optical_link_margin(power_budget, loss_budget)

PowerBudget(budget=20.1, safety_margin=2)
Power budget is 20.1dB, and loss budget is 17.9dB.
The net budget is 2.2dB and the margin of the safety factor is 0.2dB
Budget is 0.2dB over the required 2dB safety margin.


2.2

### Safety Margin/Unallocated Design (Aging Cushion) Example

$< 0\text{ dB}$ (Fail): Excessive channel attenuation. The link will drop frames or fail to establish an optical link state entirely.

In [247]:
tx = Transmitter(launch_power_dbm=-4)
rx = Receiver(receive_sensitivity_dbm=-25)

power_budget = optical_power_budget(tx, rx)
print(power_budget)

cables = [Cable(cable_length_m=49, cable_loss_db_km=0.37)]
splices = [Splice() for splice in range(21)]
connectors = [Connector() for connector in range(2)]

loss_budget = optical_loss_budget(cables, splices, connectors)
print(loss_budget)

optical_link_margin(power_budget, loss_budget)

PowerBudget(budget=21, safety_margin=2)
LossBudget(budget=21.23)
Power budget is 21dB, and loss budget is 21.23dB.
The net budget is -0.23dB and the margin of the safety factor is -2.23dB
Budget is -2.23dB under the required 2dB safety margin.


Exception: 